# YOLO11 Training & Evaluation Pipeline for Roboflow Datasets
### Training on Roboflow Export: `architecture_bom.yolo26.zip` (206 Annotated Images)

This notebook provides a fully automated pipeline to train a YOLO11 model on your custom Roboflow dataset. It mounts Google Drive, extracts the zip archive, dynamically determines if the labels are for **Object Detection** or **Instance Segmentation**, updates paths in `data.yaml` automatically, executes training with checkpoint resume/fallback, outputs evaluation curves, and provides a file uploader to test the model.

## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install YOLO, ONNX, and plotting dependencies
!pip install -q ultralytics onnx onnxslim onnxruntime pandas opencv-python pyyaml matplotlib

import os
import sys
import shutil
import torch
from pathlib import Path
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Please switch Colab Runtime to GPU for training.")

## 2. Locate and Extract Dataset (`architecture_bom.yolo26.zip`)
Searches local storage and Google Drive for the zip file. If not found, it prompts you to upload the file directly.

In [ ]:
from google.colab import files

zip_path = None
candidates = [
    "/content/architecture_bom.yolo26.zip",
    "/content/drive/MyDrive/BOM_Project/architecture_bom.yolo26.zip",
    "/content/drive/MyDrive/architecture_bom.yolo26.zip"
]

for c in candidates:
    if os.path.exists(c):
        zip_path = c
        break
        
if zip_path is None:
    print("architecture_bom.yolo26.zip was not found in Drive. Please upload the dataset zip directly:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if "architecture_bom.yolo26" in fn or fn.endswith('.zip'):
            zip_path = os.path.join("/content", fn)
            break

if zip_path and os.path.exists(zip_path):
    local_dir = "/content/architecture_bom_yolo26"
    if os.path.exists(local_dir): shutil.rmtree(local_dir)
    print(f"Unpacking dataset zip from {zip_path}...")
    shutil.unpack_archive(zip_path, local_dir, "zip")
    print("Extraction complete!")
else:
    print("ERROR: No dataset zip file could be located.")

## 3. Auto-Detect Task (Detection vs. Segmentation) & Fix YAML Paths
Parses the Roboflow folder, determines whether the annotations represent bounding boxes (Object Detection) or polygons (Instance Segmentation), and dynamically rewrites `data.yaml` with the correct root directory path.

In [ ]:
import glob
import yaml

# 1. Find data.yaml in unzipped files
yaml_files = glob.glob("/content/architecture_bom_yolo26/**/data*.yaml", recursive=True)
yaml_files.extend(glob.glob("/content/architecture_bom_yolo26/**/dataset*.yaml", recursive=True))

if not yaml_files:
    raise FileNotFoundError("data.yaml not found in unzipped files!")
    
yaml_path = yaml_files[0]
dataset_root = os.path.dirname(yaml_path)
print(f"Detected Dataset Root: {dataset_root}")

# 2. Auto-detect if Bounding Boxes or Polygons are used
label_files = glob.glob(f"{dataset_root}/**/labels/**/*.txt", recursive=True)
is_segmentation = False

if label_files:
    for lf in label_files:
        if os.path.getsize(lf) > 0:
            with open(lf, "r") as f:
                line_parts = f.readline().strip().split()
                if len(line_parts) > 5:
                    is_segmentation = True
                    break
                    
task_name = "segment" if is_segmentation else "detect"
model_weights = "yolo11s-seg.pt" if is_segmentation else "yolo11s.pt"
print(f"Auto-detected Task: {task_name.upper()} (Model: {model_weights})")

# 3. Update paths inside YAML file
with open(yaml_path, "r") as f:
    yaml_data = yaml.safe_load(f)

yaml_data["path"] = dataset_root

with open(yaml_path, "w") as f:
    yaml.safe_dump(yaml_data, f)
print(f"Dynamically updated paths in YAML: {yaml_path}")

## 4. Train YOLO Model on Roboflow Dataset

In [ ]:
from ultralytics import YOLO

# Drive models directory where everything is saved directly
drive_model_dir = "/content/drive/MyDrive/BOM_Project/roboflow_models"
os.makedirs(drive_model_dir, exist_ok=True)
resume_path = os.path.join(drive_model_dir, "yolo11_roboflow", "weights", "last.pt")

# Check auto-resume
resume_mode = False
if Path(resume_path).exists():
    print(f"Checkpoint detected at {resume_path}. Resuming training...")
    model = YOLO(resume_path)
    resume_mode = resume_path
else:
    print(f"Starting fresh training from baseline {model_weights}...")
    model = YOLO(model_weights)

print("Starting training on 206 Roboflow annotated images...")
try:
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=1024,
        batch=16,
        device=0,
        cache=True,
        amp=True,
        workers=2,
        project=drive_model_dir,
        name="yolo11_roboflow",
        plots=True,
        save=True,
        save_period=5,     # Save checkpoints every 5 epochs
        optimizer="AdamW",
        cos_lr=True,
        resume=resume_mode
    )
except Exception as e:
    print(f"Training error: {e}. Retrying with batch=8 memory fallback...")
    if resume_mode:
        model = YOLO(resume_path)
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=1024,
        batch=8,
        device=0,
        cache=True,
        amp=True,
        workers=2,
        project=drive_model_dir,
        name="yolo11_roboflow",
        plots=True,
        save=True,
        save_period=5,
        optimizer="AdamW",
        cos_lr=True,
        resume=resume_mode
    )

# Save training directory dynamically
run_dir = str(results.save_dir)
print("Training directory returned:", run_dir)
with open("/content/run_dir_roboflow.txt", "w") as f:
    f.write(run_dir)

## 5. Audit Check: Find results.csv and best.pt

In [ ]:
from pathlib import Path

print("Searching for results.csv...")
for p in Path("/content").rglob("results.csv"):
    print(p)
    
print("\nSearching for best.pt...")
for p in Path("/content").rglob("best.pt"):
    print(p)

## 6. View Evaluation Metrics & Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

if os.path.exists("/content/run_dir_roboflow.txt"):
    with open("/content/run_dir_roboflow.txt", "r") as f:
        run_dir = f.read().strip()
else:
    run_dir = os.path.join(drive_model_dir, "yolo11_roboflow")

csv_path = os.path.join(run_dir, "results.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    fig, axs = plt.subplots(1, 2, figsize=(16, 5))
    
    axs[0].plot(df['epoch'], df['train/box_loss'], label='train/box_loss')
    axs[0].plot(df['epoch'], df['val/box_loss'], label='val/box_loss')
    axs[0].set_title('Box Loss')
    axs[0].legend()
    
    # Check if segmentation or detection metric exists
    map_key = 'metrics/mAP50(B)' if 'metrics/mAP50(B)' in df.columns else 'metrics/mAP50'
    if map_key in df.columns:
        axs[1].plot(df['epoch'], df[map_key], label='mAP50 Box')
    if 'metrics/mAP50(M)' in df.columns:
        axs[1].plot(df['epoch'], df['metrics/mAP50(M)'], label='mAP50 Mask')
    axs[1].set_title('mAP Performance')
    axs[1].legend()
    plt.show()
else:
    print(f"results.csv not found at {csv_path}")

for curves in ["confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    p = os.path.join(run_dir, curves)
    if os.path.exists(p):
        print(f"\n=== {curves} ===")
        display(Image(filename=p))

## 7. Model Validation Report

In [ ]:
best_weights = os.path.join(run_dir, "weights", "best.pt")
if os.path.exists(best_weights):
    val_model = YOLO(best_weights)
    val_results = val_model.val()
    
    print("\n" + "="*60)
    print("Validation Summary:")
    print("="*60)
    for k, v in val_results.results_dict.items():
        print(f"{k:<25} {v:.4f}")
    print("="*60)
else:
    print(f"ERROR: best.pt weights not found at: {best_weights}")

## 8. Interactive Test Cell: Upload Floorplan and Run Model
Run this cell to upload any floorplan image (PNG/JPG) from your computer and visualize predictions, counts, and segment overlays instantly.

In [ ]:
from google.colab import files
import cv2
import numpy as np
from IPython.display import Image, display

best_weights = os.path.join(run_dir, "weights", "best.pt")

if not os.path.exists(best_weights):
    print("ERROR: Trained best.pt weights not found.")
else:
    print("Trained model loaded and ready. Please upload an image to test:")
    uploaded = files.upload()
    
    test_model = YOLO(best_weights)
    
    for fn in uploaded.keys():
        print(f"\nProcessing file: {fn}")
        res = test_model.predict(source=fn, imgsz=1024, conf=0.25)[0]
        
        # Get counts of detections grouped by classes
        class_names = test_model.names
        counts = {name: 0 for name in class_names.values()}
        
        if res.boxes is not None:
            for box in res.boxes:
                cls_id = int(box.cls[0].item())
                cls_name = class_names.get(cls_id, f"class_{cls_id}")
                counts[cls_name] = counts.get(cls_name, 0) + 1
                
        print(f"\n" + "-"*30 + " Detections Summary " + "-"*30)
        for name, count in counts.items():
            print(f"{name:<20} {count}")
        print(f"Total items detected: {sum(counts.values())}")
        print("-"*80)
        
        plotted_img = res.plot()
        output_filename = f"predicted_{fn}"
        cv2.imwrite(output_filename, plotted_img)
        
        print(f"Predicted Overlay ({output_filename}):")
        display(Image(filename=output_filename))

## 9. Final Google Drive Backup

In [ ]:
drive_backup = "/content/drive/MyDrive/BOM_Project/roboflow_final_run"
print(f"Backing up final run from {run_dir} to Drive: {drive_backup}...")
try:
    shutil.copytree(run_dir, drive_backup, dirs_exist_ok=True)
    print("Drive backup completed successfully!")
except Exception as e:
    print(f"Warning: Copy failed: {e}")